# Lezione 13 — Valutare un classificatore sul serio

**Come si usa questo notebook.** Come sempre. Prerequisiti: Lezioni 9-12.
Chiude la Fase 2.

**Cosa saprai fare alla fine:** giudicare un classificatore oltre
l'accuratezza — precision, recall, F1, confusion matrix, calibrazione — e
capire *quale* errore sta facendo, non solo *quanto* sbaglia.

## Parte 1 — Teoria: perché l'accuratezza non basta

L'accuratezza risponde a una sola domanda: "che frazione ho azzeccato?". Ma
nasconde **come** sbagli, e in molti problemi il *come* è tutto:

- una malattia colpisce l'1% dei pazienti: il modello che dice sempre
  "sano" ha il 99% di accuratezza ed è **inutile**;
- nel Memory AI Lab, confondere `preference` con `episodic` fa danni
  diversi che confondere `semantic` con `unknown`.

Le domande giuste, per ogni classe:

- **Precision** — *quando il modello dice "preference", quante volte ha
  ragione?* (`veri positivi / tutti i predetti positivi`). Bassa precision
  = falsi allarmi.
- **Recall** — *di tutte le vere "preference", quante ne trova?*
  (`veri positivi / tutti i positivi reali`). Basso recall = casi persi.
- **F1** — la media armonica delle due: alta solo se lo sono entrambe.

Precision e recall sono in **tensione**: alzare la soglia per "dire
preference" aumenta la precision e abbassa il recall, e viceversa. La
scelta del punto di equilibrio non è tecnica, è di **prodotto**: costa di
più un falso allarme o un caso perso? (In medicina: un esame in più o una
diagnosi mancata?)

E la **confusion matrix** è la mappa completa: righe = verità, colonne =
predizione. La diagonale sono i successi; ogni cella fuori diagonale è una
*specifica* confusione, con la sua storia.

## Parte 2 — Teoria: la calibrazione

Un classificatore softmax non emette solo la classe: emette **probabilità**
(Lezione 9). Un modello è **calibrato** se quelle probabilità dicono la
verità: tra tutte le volte che dice "80% sicuro", ha ragione circa l'80%
delle volte.

Perché conta: nel Memory AI Lab le probabilità decideranno *soglie* (sotto
quale confidenza chiedere conferma? quando scartare una classificazione?).
Con un modello sovraconfidente le soglie mentono. Le reti neurali moderne
tendono a essere **sovraconfidenti** — il controllo minimo, che facciamo
oggi: confrontare la confidenza media con l'accuratezza reale.

## Parte 3 — Il progetto: la pagella del classificatore

Riaddestriamo il modello finale della Lezione 12 (stesso seed, stessa
ricetta: il notebook resta autosufficiente) e produciamo la sua pagella
completa su **validation** — il test è già stato consumato nella Lezione 12
e resta chiuso (Lezione 3: ogni sguardo lo consuma).

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import pandas as pd
import keras

keras.utils.set_random_seed(42)
print('Keras', keras.__version__)

In [ ]:
from pathlib import Path

processed = Path('..') / 'datasets' / 'processed'
insiemi = {n: pd.read_csv(processed / f'memory_features_{n}.csv') for n in ['train', 'val']}
X = {n: f.drop(columns='target').to_numpy().astype('float32') for n, f in insiemi.items()}
y = {n: f['target'].to_numpy() for n, f in insiemi.items()}
classi = ['episodic', 'preference', 'semantic', 'unknown']

modello = keras.Sequential([
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(4, activation='softmax'),
])
modello.compile(loss='sparse_categorical_crossentropy', optimizer='adam',
                metrics=['accuracy'])
modello.fit(X['train'], y['train'], epochs=300,
            validation_data=(X['val'], y['val']),
            callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=20,
                                                     restore_best_weights=True)],
            verbose=0)
probabilita = modello.predict(X['val'], verbose=0)
predizioni = probabilita.argmax(axis=1)
print(f"accuratezza validation: {(predizioni == y['val']).mean():.0%}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y['val'], predizioni, labels=range(4),
                            target_names=classi, zero_division=0))

In [ ]:
import matplotlib.pyplot as plt

matrice = confusion_matrix(y['val'], predizioni, labels=range(4))
fig, asse = plt.subplots(figsize=(5, 4))
im = asse.imshow(matrice, cmap='Blues')
asse.set_xticks(range(4), classi, rotation=30)
asse.set_yticks(range(4), classi)
asse.set_xlabel('predetto')
asse.set_ylabel('vero')
for i in range(4):
    for j in range(4):
        asse.text(j, i, matrice[i, j], ha='center', va='center')
plt.title('Confusion matrix (validation)')
plt.tight_layout()
plt.show()

Leggi la matrice come una mappa degli errori: ogni cella fuori diagonale
racconta una confusione specifica ("le semantic scambiate per episodic").
Con 20 esempi di validation i numeri sono piccoli — la lettura giusta è
qualitativa: *dove* si concentra l'errore, non i decimali (Lezione 12: su
campioni piccoli si dichiara l'incertezza).

E il controllo di calibrazione minimo:

In [ ]:
confidenza_media = probabilita.max(axis=1).mean()
accuratezza = (predizioni == y['val']).mean()
print(f'confidenza media dichiarata : {confidenza_media:.0%}')
print(f'accuratezza reale           : {accuratezza:.0%}')
if confidenza_media > accuratezza + 0.05:
    print('-> il modello e\' SOVRACONFIDENTE: le sue probabilita\' promettono troppo')
else:
    print('-> confidenza e accuratezza sono compatibili')

## Parte 4 — Esercizio guidato

Un rilevatore di guasti su 1000 macchine: 20 guaste. Il modello A le
predice **tutte sane**; il modello B trova 15 guasti veri, ne perde 5, e
segnala 30 falsi allarmi.

Il tuo compito, a mano (carta o cella): accuratezza di A, e per B
precision, recall e F1 della classe "guasto". Poi rispondi: quale modello
sceglieresti, e quale metrica l'ha rivelato?

**Prova tu:**

In [ ]:
# A: predice 1000 volte 'sano' su 980 sani + 20 guasti
# B: 15 veri positivi, 5 falsi negativi, 30 falsi positivi

pass

### Soluzione spiegata

- A: accuratezza 980/1000 = **98%**, ma recall sui guasti = 0/20 = **0%**:
  non trova nessun guasto. L'accuratezza lo premia, il recall lo smaschera.
- B: precision = 15/(15+30) = **33%**, recall = 15/20 = **75%**,
  F1 = 2·(0.33·0.75)/(0.33+0.75) ≈ **0.46**.
- B è enormemente più utile (trova 3 guasti su 4) al prezzo di falsi
  allarmi. Se un guasto perso costa più di 30 ispezioni inutili, B vince —
  ed è il recall, non l'accuratezza, la metrica che l'ha detto.

In [ ]:
precision_B = 15 / (15 + 30)
recall_B = 15 / 20
f1_B = 2 * precision_B * recall_B / (precision_B + recall_B)
print('A: accuratezza 98%, recall guasti 0%')
print(f'B: precision {precision_B:.0%}, recall {recall_B:.0%}, F1 {f1_B:.2f}')

## Cosa hai imparato — e la Fase 2 è completa

- L'accuratezza nasconde *come* sbagli; **precision** (falsi allarmi) e
  **recall** (casi persi) lo rivelano, classe per classe; **F1** le fonde.
- Il punto di equilibrio precision/recall è una decisione di prodotto sui
  costi degli errori, non un dettaglio tecnico.
- La **confusion matrix** è la mappa: ogni cella fuori diagonale è una
  confusione specifica da investigare.
- La **calibrazione** misura se le probabilità dicono la verità: reti
  sovraconfidenti producono soglie che mentono.

## Quiz

1. Un filtro antispam ha precision 99% e recall 60% sulla classe "spam".
   Cosa succede nella casella dell'utente?
2. Perché la pagella del progetto è su validation e non sul test?
3. Confidenza media 95%, accuratezza 70%: cosa NON puoi fare con le
   probabilità di questo modello?

<details>
<summary><b>Apri le risposte</b></summary>

1. Quasi niente finisce per errore nello spam (precision alta), ma il 40%
   dello spam passa in inbox (recall basso): pochi falsi allarmi, molti
   casi persi.
2. Il test è stato consumato dalla valutazione finale della Lezione 12:
   ogni ulteriore sguardo lo trasformerebbe in un validation (Lezione 3).
   Le analisi diagnostiche si fanno su validation.
3. Usarle come soglie decisionali: "accetta se confidenza > 90%" farebbe
   passare moltissimi errori, perché il 95% dichiarato vale in realta'
   ~70%. Prima serve ricalibrare (o non fidarsi delle probabilita').
</details>

## Fonti

- scikit-learn, *Precision, recall and F-measures*:
  https://scikit-learn.org/stable/modules/model_evaluation.html#precision-recall-f-measure-metrics
- scikit-learn, *Confusion matrix*:
  https://scikit-learn.org/stable/modules/model_evaluation.html#confusion-matrix
- Guo et al. (2017), *On Calibration of Modern Neural Networks* (le reti
  moderne tendono alla sovraconfidenza): https://arxiv.org/abs/1706.04599

## Prossima lezione

Finora i dati entrano nel modello come array già in memoria. I sistemi veri
usano **pipeline di input**: `tf.data`, l'ultimo attrezzo della fase dati.
Lezione 14.